# Pandas for Machine Learning

**What is Pandas?**  
Pandas is the go-to library for loading, cleaning, and exploring data before training any ML model. If NumPy is the engine, Pandas is the garage where you prepare the car.

**Why does it matter?**  
In real ML projects, 80% of the work is data preparation — not model training. Pandas is how you do that. Every Kaggle notebook, every industry ML pipeline starts with Pandas.

**What you'll learn:**
- Series and DataFrame — the two core structures
- Loading data
- Exploring data
- Cleaning — missing values, duplicates, wrong types
- Filtering and selecting
- Grouping and aggregating
- Feature engineering with Pandas
- Converting to NumPy for ML

**Prerequisites:** 01_numpy.ipynb

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('Pandas version:', pd.__version__)

## 1. Series and DataFrame

Pandas has two core structures:
- **Series** — a 1D labelled array (one column)
- **DataFrame** — a 2D labelled table (your dataset)

In [ ]:
# Series — one feature column
ages = pd.Series([25, 30, 35, 28, 42], name='age')
print('Series:')
print(ages)
print('Type:', type(ages))
print('Values:', ages.values)      # converts to NumPy array
print('Index:', ages.index)

# DataFrame — full dataset
data = {
    'age':    [25, 30, 35, 28, 42],
    'salary': [50000, 70000, 90000, 60000, 110000],
    'city':   ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad'],
    'hired':  [1, 1, 0, 1, 0]
}

df = pd.DataFrame(data)
print('\nDataFrame:')
print(df)

## 2. Loading Data

In real projects you load from CSV, Excel, or SQL — not create manually.

In [ ]:
# Load from CSV — most common in ML
# df = pd.read_csv('data.csv')

# Load from URL — Titanic dataset
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

print('Shape:', df.shape)           # (rows, columns)
print('\nFirst 5 rows:')
print(df.head())

print('\nLast 3 rows:')
print(df.tail(3))

print('\nColumn names:', df.columns.tolist())

## 3. Exploring Data

Always explore before training. These are the first commands you run on any new dataset.

In [ ]:
# Overview
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)

print('\nInfo:')
df.info()

In [ ]:
# Statistics
print('Summary statistics:')
print(df.describe())

In [ ]:
# Missing values — critical to check
print('Missing values per column:')
print(df.isnull().sum())

print('\nMissing percentage:')
print((df.isnull().sum() / len(df) * 100).round(2))

In [ ]:
# Value counts — distribution of categorical columns
print('Survived counts:')
print(df['Survived'].value_counts())

print('\nPclass counts:')
print(df['Pclass'].value_counts())

print('\nSex counts:')
print(df['Sex'].value_counts())

## 4. Selecting Data

Two ways to select: `[]` for columns, `.loc` for label-based, `.iloc` for position-based.

In [ ]:
# Select one column — returns Series
print(df['Age'].head())

# Select multiple columns — returns DataFrame
print(df[['Age', 'Fare', 'Survived']].head())

# loc — label based (inclusive on both ends)
print('\nloc rows 0-3, specific columns:')
print(df.loc[0:3, ['Name', 'Age', 'Survived']])

# iloc — position based (exclusive on end like Python)
print('\niloc first 3 rows, first 4 columns:')
print(df.iloc[:3, :4])

## 5. Filtering

In ML you often need to filter rows based on conditions — removing outliers, selecting subsets.

In [ ]:
# Single condition
survivors = df[df['Survived'] == 1]
print('Survivors:', len(survivors))

# Multiple conditions — use & and | not 'and' 'or'
women_survived = df[(df['Sex'] == 'female') & (df['Survived'] == 1)]
print('Women who survived:', len(women_survived))

# isin — filter by list of values
first_second = df[df['Pclass'].isin([1, 2])]
print('First and second class:', len(first_second))

# Remove outliers — age below 100
df_clean = df[df['Age'] < 100]
print('After removing age outliers:', len(df_clean))

## 6. Cleaning Data

Real data is always messy. This is where most ML project time goes.

In [ ]:
df_clean = df.copy()    # always work on a copy

# Handle missing values
# Option 1 — drop rows with missing values
df_dropped = df_clean.dropna()
print('After dropping NaN rows:', df_dropped.shape)

# Option 2 — fill with mean (numeric columns)
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())
print('Age NaNs after fill:', df_clean['Age'].isnull().sum())

# Option 3 — fill with mode (categorical columns)
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])
print('Embarked NaNs after fill:', df_clean['Embarked'].isnull().sum())

# Drop columns with too many missing values
df_clean = df_clean.drop(columns=['Cabin'])   # 77% missing — not useful

# Drop duplicates
df_clean = df_clean.drop_duplicates()
print('\nShape after cleaning:', df_clean.shape)

## 7. Grouping and Aggregating

Used to understand patterns in data — survival rate by class, average salary by department, etc.

In [ ]:
# Survival rate by class
print('Survival rate by Pclass:')
print(df.groupby('Pclass')['Survived'].mean().round(2))

# Survival rate by sex
print('\nSurvival rate by Sex:')
print(df.groupby('Sex')['Survived'].mean().round(2))

# Multiple aggregations
print('\nAge stats by Pclass:')
print(df.groupby('Pclass')['Age'].agg(['mean', 'median', 'std']).round(2))

# Pivot table
pivot = df.pivot_table(values='Survived',
                       index='Pclass',
                       columns='Sex',
                       aggfunc='mean').round(2)
print('\nSurvival rate by Pclass and Sex:')
print(pivot)

## 8. Feature Engineering with Pandas

Creating new features from existing ones. This often improves model performance more than tuning the model itself.

In [ ]:
df_feat = df_clean.copy()

# Create new features
df_feat['FamilySize'] = df_feat['SibSp'] + df_feat['Parch'] + 1
df_feat['IsAlone']    = (df_feat['FamilySize'] == 1).astype(int)
df_feat['FarePerPerson'] = df_feat['Fare'] / df_feat['FamilySize']

# Age groups — bin continuous into categories
df_feat['AgeGroup'] = pd.cut(df_feat['Age'],
                              bins=[0, 12, 18, 35, 60, 100],
                              labels=['Child', 'Teen', 'Adult', 'MiddleAge', 'Senior'])

# Extract title from name
df_feat['Title'] = df_feat['Name'].str.extract(r' ([A-Za-z]+)\.')

print(df_feat[['Name', 'Title', 'Age', 'AgeGroup', 'FamilySize', 'IsAlone']].head(10))

In [ ]:
# Encode categorical columns — ML models need numbers

# Label encoding — for binary categories
df_feat['Sex_encoded'] = df_feat['Sex'].map({'male': 0, 'female': 1})

# One-hot encoding — for multi-class categories
embarked_dummies = pd.get_dummies(df_feat['Embarked'], prefix='Embarked')
df_feat = pd.concat([df_feat, embarked_dummies], axis=1)

print('Encoded columns added:')
print(df_feat[['Sex', 'Sex_encoded', 'Embarked', 'Embarked_C', 'Embarked_Q', 'Embarked_S']].head())

## 9. Converting to NumPy for ML

ML models take NumPy arrays, not DataFrames. This is the final step before training.

In [ ]:
# Select final features
feature_cols = ['Pclass', 'Sex_encoded', 'Age', 'FamilySize',
                'IsAlone', 'Fare', 'Embarked_C', 'Embarked_Q', 'Embarked_S']

X = df_feat[feature_cols].values    # .values converts to NumPy
y = df_feat['Survived'].values

print('X type:', type(X))           # numpy.ndarray
print('X shape:', X.shape)
print('y shape:', y.shape)
print('y unique values:', np.unique(y))

# Alternatively
X_alt = df_feat[feature_cols].to_numpy()
print('\nSame result with to_numpy():', X_alt.shape)

---

## Mini Project — Full EDA on Titanic Dataset

**Task:** Explore the Titanic dataset end-to-end and answer: what factors most influenced survival?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. Survival distribution
df['Survived'].value_counts().plot(kind='bar', ax=axes[0,0],
                                    color=['salmon', 'steelblue'])
axes[0,0].set_title('Survival Count')
axes[0,0].set_xticklabels(['Died', 'Survived'], rotation=0)

# 2. Survival by sex
df.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[0,1],
                                           color=['steelblue', 'salmon'])
axes[0,1].set_title('Survival Rate by Sex')
axes[0,1].set_ylabel('Survival Rate')
axes[0,1].tick_params(axis='x', rotation=0)

# 3. Survival by class
df.groupby('Pclass')['Survived'].mean().plot(kind='bar', ax=axes[0,2],
                                              color='steelblue')
axes[0,2].set_title('Survival Rate by Class')
axes[0,2].set_ylabel('Survival Rate')
axes[0,2].tick_params(axis='x', rotation=0)

# 4. Age distribution
df['Age'].dropna().plot(kind='hist', bins=30, ax=axes[1,0], color='steelblue')
axes[1,0].set_title('Age Distribution')
axes[1,0].set_xlabel('Age')

# 5. Fare distribution
df['Fare'].plot(kind='hist', bins=40, ax=axes[1,1], color='green')
axes[1,1].set_title('Fare Distribution')
axes[1,1].set_xlabel('Fare')

# 6. Age vs Fare
survived     = df[df['Survived'] == 1]
not_survived = df[df['Survived'] == 0]
axes[1,2].scatter(not_survived['Age'], not_survived['Fare'],
                  alpha=0.4, label='Died', color='salmon', s=10)
axes[1,2].scatter(survived['Age'], survived['Fare'],
                  alpha=0.4, label='Survived', color='steelblue', s=10)
axes[1,2].set_title('Age vs Fare by Survival')
axes[1,2].set_xlabel('Age')
axes[1,2].set_ylabel('Fare')
axes[1,2].legend()

plt.suptitle('Titanic Dataset — Exploratory Data Analysis', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Key findings
print('Key Findings from EDA')
print('---------------------')
print(f"Overall survival rate: {df['Survived'].mean()*100:.1f}%")
print(f"Female survival rate:  {df[df['Sex']=='female']['Survived'].mean()*100:.1f}%")
print(f"Male survival rate:    {df[df['Sex']=='male']['Survived'].mean()*100:.1f}%")
print(f"1st class survival:    {df[df['Pclass']==1]['Survived'].mean()*100:.1f}%")
print(f"2nd class survival:    {df[df['Pclass']==2]['Survived'].mean()*100:.1f}%")
print(f"3rd class survival:    {df[df['Pclass']==3]['Survived'].mean()*100:.1f}%")
print(f"Average age survived:  {df[df['Survived']==1]['Age'].mean():.1f}")
print(f"Average age died:      {df[df['Survived']==0]['Age'].mean():.1f}")

---

## Common Mistakes to Avoid

1. **Not copying before modifying** — always `df_clean = df.copy()` before making changes. Otherwise you modify the original.
2. **Dropping NaN rows blindly** — if 20% of rows have missing age, dropping them loses too much data. Fill instead.
3. **One-hot encoding the target** — only encode features, never the target column.
4. **Leaking test data into scaling** — compute statistics on training data only.
5. **Forgetting `.values`** — many sklearn functions expect NumPy arrays. Always convert with `.values` or `.to_numpy()`.
6. **Using `&` and `|` not `and` and `or`** — Python's `and`/`or` don't work with Pandas boolean arrays.

---

## Interview Questions

1. What is the difference between `loc` and `iloc`?
2. How do you handle missing values in a dataset? When do you drop vs fill?
3. What is one-hot encoding and when do you use it?
4. What is the difference between `groupby` and `pivot_table`?
5. Why should you always work on a copy of a DataFrame?
6. How do you convert a Pandas DataFrame to a NumPy array?
7. What is feature engineering and give an example from the Titanic dataset?

---

**Next:** `03_matplotlib_seaborn.ipynb` — visualising data before and after training